## Action Responder Prompt

This notebook hopes to:

- Gather trace data for action-responder agent
- Structure and save as mlflow dataset
- Utilize judges to evaluate current agent
- Try with better model to confirm judges are working
- Run GEPA optimzation to improve

In [1]:
import mlflow
from mlflow.tracking import MlflowClient

from summit_sim.settings import settings

mlflow.tracing.enable_notebook_display()

client = MlflowClient()
mlflow.set_tracking_uri(settings.mlflow_tracking_uri)
mlflow.set_experiment(settings.mlflow_experiment_name)
experiment = client.get_experiment_by_name(settings.mlflow_experiment_name)


# Get all traces from last 7 days (adjust as needed)
filter_string = """
tags.graph_type = 'simulation' AND
tags.agent_name = 'action-responder'
"""
traces = client.search_traces(
    locations=[experiment.experiment_id],
    filter_string=filter_string,
    max_results=500,
)

2026-03-29 21:25:07 [WARNING] urllib3.connectionpool: Connection pool is full, discarding connection: mlflow.bhamm-lab.com. Connection pool size: 10
2026-03-29 21:25:07 [WARNING] urllib3.connectionpool: Connection pool is full, discarding connection: mlflow.bhamm-lab.com. Connection pool size: 10
2026-03-29 21:25:07 [WARNING] urllib3.connectionpool: Connection pool is full, discarding connection: mlflow.bhamm-lab.com. Connection pool size: 10
2026-03-29 21:25:07 [WARNING] urllib3.connectionpool: Connection pool is full, discarding connection: mlflow.bhamm-lab.com. Connection pool size: 10


In [2]:
spans = [
    span
    for trace in traces
    for span in client.get_trace(trace.info.trace_id).data.spans
    if span.name == "action_response_agent"
]

dataset = [{"inputs": span.inputs, "outputs": span.outputs} for span in spans]

print(f"Found {len(dataset)} action_response_agent spans")

spans

Found 24 action_response_agent spans


[Span(name='action_response_agent', trace_id='tr-affe256ef8a85223f22a7b2f6f8953f9', span_id='dbbb9f7cd9a59114', parent_id='9029006bcb84b67a'),
 Span(name='action_response_agent', trace_id='tr-02676ba69b9f82ffdc08ac94d18f06e2', span_id='2189d23417dddc2b', parent_id='0f9bcffcde27ea61'),
 Span(name='action_response_agent', trace_id='tr-141e739e58049c1a79786b59ee8fdd03', span_id='8b4960e90c0953f5', parent_id='1c49ba6e68b0f081'),
 Span(name='action_response_agent', trace_id='tr-5edc75e07690dd965b51d8b35fb26a86', span_id='1e7a3d0bf0f26664', parent_id='4453717196ef353f'),
 Span(name='action_response_agent', trace_id='tr-2ea58049cc996b1cccb1815df46cec62', span_id='72923819e572101d', parent_id='3c7c52bc6d95e7d2'),
 Span(name='action_response_agent', trace_id='tr-fa63eaabbc1080b664d1dc4aef22e720', span_id='9cef0743cdbb6ffa', parent_id='77a29bef3bd1938d'),
 Span(name='action_response_agent', trace_id='tr-6979387d5e89563634062b84bb2f547c', span_id='81d6190fd3b6bdad', parent_id='11df237beb6fd948'),

[Trace(trace_id=tr-affe256ef8a85223f22a7b2f6f8953f9), Trace(trace_id=tr-02676ba69b9f82ffdc08ac94d18f06e2), Trace(trace_id=tr-141e739e58049c1a79786b59ee8fdd03), Trace(trace_id=tr-5edc75e07690dd965b51d8b35fb26a86), Trace(trace_id=tr-2ea58049cc996b1cccb1815df46cec62), Trace(trace_id=tr-fa63eaabbc1080b664d1dc4aef22e720), Trace(trace_id=tr-6979387d5e89563634062b84bb2f547c), Trace(trace_id=tr-35573d4c1b256e7ca341b17d06b2f062), Trace(trace_id=tr-79f172b56f8eddf75effd7bc01b520e1), Trace(trace_id=tr-050dbc90371a14aa2798144552bef0ac)]

In [ ]:
from mlflow.genai.judges import make_judge

JUDGE_MODEL_ENDPOINT = "openrouter:/anthropic/claude-haiku-4.5"

COMPLETION_JUDGE_INSTRUCTIONS = """\
You are evaluating the scoring accuracy and pedagogical quality of an
AI-generated response in a wilderness first responder (WFR) training simulation.

=== PATIENT ASSESSMENT SYSTEM (PAS) - GUIDELINES ===

The PAS follows this general order, but students may bundle steps efficiently or adapt based on the situation:

1. SCENE SIZE-UP: 0-.2 points
   - Check for hazards (environmental dangers, unstable terrain, etc.)
   - Identify mechanism of injury (MOI)
   - Count patients and assess available resources

2. PRIMARY ASSESSMENT (ABCDE): 0-.2 points
   - A: Airway assessment
   - B: Breathing evaluation
   - C: Circulation (pulse, bleeding, shock signs)
   - D: Disability (mental status, AVPU)
   - E: Exposure/Environment (clothing, temperature, elements)

3. SECONDARY ASSESSMENT: 0-.2 points
   - Vital signs (HR, RR, BP, SCTM, temperature)
   - Head-to-Toe exam (systematic physical check)
   - SAMPLE history (Signs/Symptoms, Allergies, Medications, Past history, Last intake, Events leading to injury)

4. TREATMENT: 0-.2 points
   - Address immediate life threats
   - Immobilize injuries
   - Wound care
   - Pain management

5. EVACUATION PLAN: 0-.2 points
   - Stay vs. Go decision
   - Resource planning
   - Timeline establishment

AI Input:
{{ inputs }}

DO NOT CONSIDER PREVIOUS INPUTS
Only evaluate the final AI agent output below:
{{ outputs }}


Evaluate against these criteria:
1. Does completion_score align with the WFR rubric based on cumulative actions?
2. Is the score increase from previous turn reasonable (<=0.2 unless explicit bundling)?
3. Does the score always build on itself? Does completion_score never decrease across turns?

Provide a score of 0-1 on how well the AI Agent performs
"""


completion_judge = make_judge(
    name="completion-judge",
    instructions=COMPLETION_JUDGE_INSTRUCTIONS,
    model=JUDGE_MODEL_ENDPOINT,
    feedback_value_type=float,
)

In [4]:
QUALITY_JUDGE_INSTRUCTIONS = """\
You are evaluating the structure and quality of an AI-generated response \
in a wilderness first responder training simulation.

AI Input:
{{ inputs }}

DO NOT CONSIDER PREVIOUS INPUTS
Only evaluate the final AI agent output below:
{{ outputs }}


Evaluate against these criteria:
1. Does narrative_text end with an open question? Does feedback contain NO questions?
2. Is the feedback personalized to the students response and focus only on history, not future?
3. Is the feedback encouraging and constructive without harsh corrections and focused
    on feedback alone?
4. Does the narrative not overtly share hidden information, only revealing
    if the student's action constitutes it?
5. Is feedback and narrative length between 2-4 sentences?

Provide a score of 0-1 on how well the AI Agent performs
"""


quality_judge = make_judge(
    name="quality-judge",
    instructions=QUALITY_JUDGE_INSTRUCTIONS,
    model=JUDGE_MODEL_ENDPOINT,
    feedback_value_type=float,
)

In [ ]:
MEDICAL_JUDGE_INSTRUCTIONS = """\
You are evaluating the medical accuracy of an AI-generated response in a \
wilderness first responder training simulation.

AI Input:
{{ inputs }}

DO NOT CONSIDER PREVIOUS INPUTS
Only evaluate the final AI agent output below:
{{ outputs }}


TIP: Reference hidden_truth and hidden_state to determine if treatment was premature.

Evaluate against these criteria:
1. Is was_correct accurate?
 - was_correct should be FALSE if student performed treatment \
    (splint, bandage, medication, move patient) without assessment
 - was_correct should be TRUE for assessment actions (vitals, exam, SAMPLE history)
2. Does was_correct accurately gage the quality of the student's action?
3. Is there anything in the AI response that is not medically accurate? If so,
    automatically provide a zero score (THE SUPERSEDES ANY PREVIOUS ASSESSMENT).

Provide a score of 0-1 on how well the AI Agent performs
"""


medical_judge = make_judge(
    name="medical-judge",
    instructions=MEDICAL_JUDGE_INSTRUCTIONS,
    model=JUDGE_MODEL_ENDPOINT,
    feedback_value_type=float,
)

In [6]:
scorers = [completion_judge, quality_judge, medical_judge]

results = mlflow.genai.evaluate(
    data=dataset,
    scorers=scorers,
)

2026/03/29 21:25:11 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.


Evaluating:   0%|          | 0/24 [Elapsed: 00:00, Remaining: ?] 

21:25:12 - LiteLLM:INFO: utils.py:3995 - 
LiteLLM completion() model= anthropic/claude-haiku-4.5; provider = openrouter
2026-03-29 21:25:12 [INFO] LiteLLM: 
LiteLLM completion() model= anthropic/claude-haiku-4.5; provider = openrouter
21:25:12 - LiteLLM:INFO: utils.py:3995 - 
LiteLLM completion() model= anthropic/claude-haiku-4.5; provider = openrouter
2026-03-29 21:25:12 [INFO] LiteLLM: 
LiteLLM completion() model= anthropic/claude-haiku-4.5; provider = openrouter
21:25:12 - LiteLLM:INFO: utils.py:3995 - 
LiteLLM completion() model= anthropic/claude-haiku-4.5; provider = openrouter
2026-03-29 21:25:12 [INFO] LiteLLM: 
LiteLLM completion() model= anthropic/claude-haiku-4.5; provider = openrouter
21:25:12 - LiteLLM:INFO: utils.py:3995 - 
LiteLLM completion() model= anthropic/claude-haiku-4.5; provider = openrouter
2026-03-29 21:25:12 [INFO] LiteLLM: 
LiteLLM completion() model= anthropic/claude-haiku-4.5; provider = openrouter
21:25:12 - LiteLLM:INFO: utils.py:3995 - 
LiteLLM completion